# RealMLP v2 2단계 본 run — 배깅+Stint-cat+arch 번들 (ep64×n_ens15, 5-fold)

docs/wiki/realmlp_v2_plan.md · 1단계 통과(fold0 0.951186 vs exp_024 0.949893, **+0.0013**).
번들: n_ens=15 배깅 + Stint_cat(5+) + yekenot arch(hidden[512,256,128]·silu·plr_sigma2.33·emb6).
게이트: 스택 swap(exp_024→v2) meta-OOF +0.0003↑ 또는 RealMLP 가중↑.

In [ ]:
# 1) input 자동탐색 (마운트 비표준: /kaggle/input/{datasets,competitions}/...) — torch import 前
import sys, os, glob, subprocess
from pathlib import Path
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
c = glob.glob('/kaggle/input/**/src/config.py', recursive=True); assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1]); print('SRC_ROOT:', SRC_ROOT)
cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True); assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0]); print('COMP:', COMP)
ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True); assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0]); print('AUG:', AUG)

In [ ]:
# 2) GPU 종류 감지(nvidia-smi, torch import 前) → 조건부 torch. 그 위에 프로젝트 deps.
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU)
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
if 'P100' in GPU:
    print('P100(sm_60) → cu121 torch trio 재설치 (Kaggle 기본 torch 는 sm_70+ 만)')
    pip('torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1',
        '--index-url','https://download.pytorch.org/whl/cu121')
else:
    print('T4 등(sm_75+) → Kaggle 기본 torch 유지')
pip('pytabkit','hydra-core','python-dotenv')

In [ ]:
# 3) torch CUDA 실연산 검증 + import 체인 fast-fail
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))
_x = torch.randn(256, 256, device='cuda'); _v = (_x @ _x).sum().item()
print('CUDA matmul OK')
sys.path.insert(0, SRC_ROOT)
from src import config
from src.train_realmlp import run
print('import OK:', config.__file__)

In [ ]:
# 4) 경로 override
config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG
out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'; config.SUBMISSION_DIR = out / 'submissions'; config.LOG_DIR = out / 'logs'
import pandas as pd
_a = pd.read_csv(config.SOURCE_AUG_PATH); print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'

In [ ]:
# 5) cfg 구성 — v2 번들: ep64 × n_ens15 + Stint-cat + yekenot arch, 5-fold
from omegaconf import OmegaConf
CONF = Path(SRC_ROOT) / 'conf'
model_cfg = OmegaConf.load(CONF / 'model' / 'realmlp.yaml')
model_cfg.params.n_epochs = 64
model_cfg.params['n_ens'] = 15
# yekenot arch 차용 (무탐색 이식, realmlp_v2_plan.md)
model_cfg.params['hidden_sizes'] = [512, 256, 128]
model_cfg.params['act'] = 'silu'
model_cfg.params['plr_sigma'] = 2.33
model_cfg.params['embedding_size'] = 6
cfg = OmegaConf.create({
    'exp_id': 'exp_032_rmlp_v2_full',
    'notes': 'v2 stage2 full: ep64 x n_ens15 + Stint_cat + yekenot arch, 5-fold. gate=stack swap vs exp_024(0.948773)',
    'use_wandb': False,
    'max_folds': None,
    'model': model_cfg,
    'features': OmegaConf.load(CONF / 'features' / 'realmlp_fe_v2.yaml'),
    'augment': {'enabled': True, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))
print('gate: stack swap exp_024->v2 meta-OOF +0.0003 or RealMLP weight up')

In [ ]:
# 6) 학습 (5-fold). 예상 ~4.5h (ep64 x n_ens15 x 5fold, P100).
import time
t0 = time.time()
result = run(cfg)
print(result, f'\n총 {time.time()-t0:.0f}s')

In [ ]:
# 7) 산출물 확인 + 결과 요약
import pandas as pd
print('cv_mean =', result.get('cv_mean'))
print('fold_scores =', result.get('fold_scores'))
print('exp_024 OOF = 0.948773 (스택 swap 기준선)')
out = Path('/kaggle/working')
for name in ['oof/exp_032_rmlp_v2_full.csv', 'submissions/exp_032_rmlp_v2_full.csv']:
    p = out / name
    if p.exists():
        d = pd.read_csv(p); print(p.name, d.shape, list(d.columns))